In [ ]:
import sys

!which {sys.executable}

In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
# %load /ut3/jerome/github/research/project/wakai/notebooks/init_ipynb
import os
import sys
import matplotlib.pyplot as pl

# sys.path.insert(0, '../code')
# assert os.environ["CONDA_DEFAULT_ENV"] == 'wakai'

plotpar = {
           'font.size': 20,
           'xtick.labelsize': 16,
           'ytick.labelsize': 16,
           'xtick.direction': 'in',
           'ytick.direction': 'in',
           'xtick.color':'black',
           'ytick.color':'black',
           'xtick.major.width':2,
           'ytick.major.width':2,
           'xtick.major.size':10,
           'ytick.major.size':10,
           'xtick.minor.width':1,
           'ytick.minor.width':1,
           'xtick.minor.size':6,
           'ytick.minor.size':6,
           'axes.labelsize': 16,
           'axes.labelcolor':'black',
           'axes.labelcolor':'black',
           'axes.spines.top':True,
           'axes.spines.right':True,
           'axes.linewidth':3,
           'axes.edgecolor':'black',
           'figure.facecolor':'none',
           'legend.facecolor':'none',
           'legend.fontsize': 16,
           'text.color':'black',
           'pdf.fonttype': 42,
           #'text.usetex': False,
           }
pl.rcParams.update(plotpar)

# Perren+2023 catalog

```
Name(s)          : Name(s) associated to the OC
name             : Formatted naming to match the members file 'main name'
N_50             : Number of estimated members with P>0.5
r_50             : Radius that contains half the members (in arcmin)
RA_ICRS          : Right ascension
DE_ICRS          : Declination
GLON             : Galactic longitude
GLAT             : Galactic latitude
Plx              : Parallax [mas]
pmRA             : Right ascension proper motion [mas/yr]
pmDE             : Declination proper motion [mas/yr]
Rv               : Radial velocity [km/s]
N_Rv             : Number of members used to estimate the radial velocity
Dist_[kpc]       : Distance [kpc]
Dist_STDDEV      : Distance standard deviation
Av_[mag]         : Absorption in the V band [mag]
Av_STDDEV]       : Absorption standard deviation
Diff_ext_[mag]   : Differential extinction [mag]
Diff_ext_STDDEV  : Differential extinction standard deviation
Age_[Myr]        : Age [Myr]
Age_STDDEV       : Age standard deviation
FeH_[dex]        : Metallicity [dex]
FeH_STDDEV       : Metallicity standard deviation
Mass_[Msun]      : Mass [Msun]
Mass_STDDEV      : Mass standard deviation
Binary_fr        : Estimated binary fraction
Binary_fr_STDDEV : Estimated binary fraction standard deviation
BSS              : Blue stragglers (total number)
BSS_STDDEV       : Blue stragglers standard deviation
C3               : Combined quality class (see Perren et al. 2023)
P_dup            : Probability that this entry is a duplicate of a previous one
UTI              : UCC Trust Index, a measure of the reliability of the cluster
bad_oc           : Flag that indicates that this is likely an asterism,
                   moving group or artifact (based on UTI value, duplication
                   probability and presence in literature)
```

In [ ]:
import pandas as pd

df = pd.read_csv('../data/Perren2023/UCC_cat.csv')
df.tail()

In [ ]:
df.shape

In [ ]:
df.columns

https://ucc.ar/faq/#what-is-the-uti-parameter

In [ ]:
idx=df['UTI']>0.9
df[idx].shape

In [ ]:
# non-negative parallax
df = df.dropna(subset=['RA_ICRS','DE_ICRS','Plx'])[df.Plx>0]
df.shape

In [ ]:
df['Name(s)'].unique()

In [ ]:
df.name.unique().shape

In [ ]:
df['Name(s)'].str.split(';').explode().unique().shape

In [ ]:
df = df.rename(
    {'Plx': 'parallax',
     'Rv': 'radial_velocity',
     'RA_ICRS': 'ra',
     'DE_ICRS': 'dec',
     'pmRA': 'pmra',
     'pmDE': 'pmdec',
    }, axis=1
)

In [ ]:
from wakai.catalog import get_transformed_coord

df = get_transformed_coord(df, frame='galactocentric')
df.head()

In [ ]:
# New row as a dictionary
sun = pd.DataFrame([{'name': 'sun', 
                     'X': -8200,
                     'Y': 0, 
                     'Z': 25,
                     'U': 11.1,
                     'V': 12.24,
                     'W': 7.25
                    }
                   ])
# Concatenate them
df = pd.concat([df, sun], ignore_index=True)

In [ ]:
import numpy as np

def get_tangential_velocity(X, Y, u, v):
    """
    Compute tangential velocity vector components.
    """
    # Convert (u,v) -> Cartesian velocities
    Vx = -u
    Vy = v

    R = np.sqrt(X**2 + Y**2)

    # Unit tangential vector
    tx = -Y / R
    ty = X / R

    # Tangential velocity magnitude
    v_tan = Vx * tx + Vy * ty

    # Tangential velocity vector
    Vtx = v_tan * tx
    Vty = v_tan * ty

    return Vtx, Vty

Vtx, Vty = get_tangential_velocity(df.X, df.Y, df.U, df.V)

df['Vtx'] = Vtx
df['Vty'] = Vty

In [ ]:
# nearby clusters
idx = df.UTI>0.9
df[idx].sort_values(by='parallax', ascending=False).reset_index(drop=True).head(10)

In [ ]:
class ReidSpiralArm:
    """
    Spiral arm model based on Reid et al. (2019):
    https://arxiv.org/abs/1908.05625
    ln(Rref/R)=−(θ−θref)tan(phi)
    
    c.f. https://arxiv.org/pdf/2209.02731
    """

    def __init__(self, name, pitch_deg, R_ref, theta_ref_deg):
        self.name = name
        self.pitch = np.radians(pitch_deg)
        self.R_ref = R_ref
        self.theta_ref = np.radians(theta_ref_deg)

    def compute(self, theta):
        """
        Compute (X, Y) coordinates of the arm.
        """
        R = self.R_ref * np.exp(-(theta - self.theta_ref) * np.tan(self.pitch))
        X = R * np.cos(theta)
        Y = R * np.sin(theta)
        return X, Y


def plot_reid_spiral_arms(theta=None, ax=None):
    if theta is None:
        theta = np.linspace(-np.pi, np.pi, 2000)
    
    arms = [
        # name, phi in deg, R in kpc, theta in deg
        ReidSpiralArm("Scutum-Centaurus", 19.8, 5.0, 25),
        ReidSpiralArm("Sagittarius", 6.9, 6.6, 25),
        ReidSpiralArm("Local (Orion)", 12.8, 8.2, 8),
        ReidSpiralArm("Perseus", 9.4, 9.9, 10),
        ReidSpiralArm("Outer", 13.8, 13.0, 15),
    ]

    if ax is None:
        fig, ax = pl.subplots(figsize=(7,7))

    for arm in arms:
        if arm.name=='Local (Orion)':
            theta0 = np.linspace(np.radians(-60), np.radians(80), 500)
        else:
            theta0 = theta
        X, Y = arm.compute(theta0)

        # Optional: restrict radial range (more realistic)
        R = np.sqrt(X**2 + Y**2)
        mask = (R > 3) & (R < 15)

        ax.plot(-X[mask], Y[mask], label=arm.name)

    # # Galactic center
    # ax.scatter(0, 0, marker='x', label='Galactic Center')

    # # Sun
    # R_sun = -8.2
    # ax.scatter(R_sun, 0, marker='o', label='Sun')

    ax.set_aspect('equal')
    ax.set_xlabel("X [kpc]")
    ax.set_ylabel("Y [kpc]")
    # pl.title("Milky Way Spiral Arms (Reid et al. 2019)")
    pl.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    # pl.grid(True)

In [ ]:
idx = df.UTI>0.5
nearby_cluster_names = df[idx].sort_values(by='parallax', ascending=False).head(10).name.unique()
nearby_cluster_names

In [ ]:
import matplotlib.pyplot as pl
from matplotlib.colors import LogNorm
skip = 1
idx = df['parallax']>0.01
# idx &= df['UTI']>0.5
# idx = df.name.isin(nearby_cluster_names)
c = 'Age_[Myr]'
idx &= df[c]<1e3
pl.scatter(df[idx].X.iloc[::skip]/1e3, 
           df[idx].Y.iloc[::skip]/1e3,
           marker='.', 
           cmap='RdBu', 
           norm=LogNorm(),
           c=df.loc[idx,c].iloc[::skip]
          )
pl.colorbar(label='Age [Myr]')
# Galactic center
pl.scatter(0, 0, marker='x', color='C3', s=200)
# Sun
s = df.query("name=='sun'").squeeze()
pl.scatter(s.X/1e3, s.Y/1e3, marker=r'$\odot$', c='C2', label='Sun', s=200)
pl.xlabel('X [kpc]')
pl.ylabel('Y [kpc]')
pl.xlim(-20, 20)
pl.ylim(-20, 20)
pl.title("Open clusters in the Milky Way", fontsize=14)

In [ ]:
df.shape  # 18k clusters

In [ ]:
import matplotlib.pyplot as pl

scale = np.nanpercentile(np.sqrt(df.Vtx**2 + df.Vty**2), 50)
skip = 2
idx = df['parallax']>0.01
# idx &= df['UTI']>0.5
# idx = df.name.isin(nearby_cluster_names)
pl.scatter(df[idx].X.iloc[::skip]/1e3, 
           df[idx].Y.iloc[::skip]/1e3,
           marker='.',
           c='pink'
          )
pl.xlim(-20, 20)
pl.ylim(-20, 20)
pl.xlabel('X [kpc]')
pl.ylabel('Y [kpc]')
pl.quiver(df[idx].X.iloc[::skip]/1e3, 
          df[idx].Y.iloc[::skip]/1e3, 
          df[idx].Vtx.iloc[::skip], 
          df[idx].Vty.iloc[::skip], 
          color='k',
          angles='xy', scale_units='xy', scale=scale)
# Galactic center
pl.scatter(0, 0, marker='x', color='C3', s=200)
# Galactic bar

def get_galactic_bar_XY(a=4000, b=1500, 
                     # angle_deg=25.0, # inner bar
                     angle_deg=45.0, # long bar
                     n_points=500):
    angle = np.radians(angle_deg)
    theta = np.linspace(0, 2*np.pi, n_points)
    
    x_prime = a * np.cos(theta)
    y_prime = b * np.sin(theta)

    # Rotate
    X = x_prime * np.cos(angle) - y_prime * np.sin(angle)
    Y = x_prime * np.sin(angle) + y_prime * np.cos(angle)

    return X, Y
    
Xb, Yb = get_galactic_bar_XY()
pl.plot(Xb/1e3, Yb/1e3, label='Galactic Bar', lw=2, ls='--', color='C3')
# Sun
s = df.query("name=='sun'").squeeze()
pl.scatter(s.X/1e3, s.Y/1e3, marker=r'$\odot$', c='C2', label='Sun', s=200)
pl.quiver(
    s.X/1e3, s.Y/1e3, #kpc
    s.Vtx/1, s.Vty/1,   # purely tangential at Sun position
    angles='xy',
    scale_units='xy',
    scale=scale,
    color='C2',
    zorder=999
)
theta = np.linspace(np.radians(-45), np.radians(135), 1000)
ax = pl.gca()
plot_reid_spiral_arms(theta=theta, ax=ax)
pl.title("Open clusters in the Milky Way", fontsize=14)

In [ ]:
import jax
import jax.numpy as jnp
from functools import partial

G = 4.30091e-3  # pc (km/s)^2 / Msun


def compute_acceleration(
    positions: jnp.ndarray,
    masses: jnp.ndarray,
    softening: float,
) -> jnp.ndarray:
    """
    Compute gravitational acceleration using direct summation.

    Parameters
    ----------
    positions : (N, 3)
    masses    : (N,)
    softening : float

    Returns
    -------
    accelerations : (N, 3)
    """
    # Pairwise separation vectors
    diff = positions[:, None, :] - positions[None, :, :]  # (N, N, 3)

    # Squared distances with softening
    dist_sq = jnp.sum(diff**2, axis=-1) + softening**2  # (N, N)

    # Inverse distance cubed
    inv_dist3 = dist_sq ** (-1.5)

    # Remove self-interaction
    inv_dist3 = jnp.where(dist_sq > 0.0, inv_dist3, 0.0)

    # Weighted sum over all particles
    acc = -G * jnp.sum(
        diff * inv_dist3[..., None] * masses[None, :, None],
        axis=1
    )

    return acc


@jax.jit
def leapfrog_step(
    positions: jnp.ndarray,
    velocities: jnp.ndarray,
    masses: jnp.ndarray,
    dt: float,
    softening: float,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    """
    Perform a single leapfrog (kick-drift-kick) step.
    """

    # --- First kick (half step) ---
    acc = compute_acceleration(positions, masses, softening)
    v_half = velocities + 0.5 * dt * acc

    # --- Drift (full step) ---
    new_positions = positions + dt * v_half

    # --- Second kick (half step) ---
    new_acc = compute_acceleration(new_positions, masses, softening)
    new_velocities = v_half + 0.5 * dt * new_acc

    return new_positions, new_velocities


@partial(jax.jit, static_argnums=(4,))
def simulate(
    positions: jnp.ndarray,
    velocities: jnp.ndarray,
    masses: jnp.ndarray,
    dt: float,
    n_steps: int,
    softening: float,
):
    """
    Run N-body simulation using JAX scan.

    Returns
    -------
    final_positions, final_velocities, trajectory
    """

    def step_fn(state, _):
        pos, vel = state
        new_pos, new_vel = leapfrog_step(pos, vel, masses, dt, softening)
        return (new_pos, new_vel), (new_pos, new_vel)

    # Time integration loop
    (final_pos, final_vel), trajectory = jax.lax.scan(
        step_fn,
        (positions, velocities),
        xs=None,
        length=n_steps,
    )

    return final_pos, final_vel, trajectory

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- Simulation ---
key = jax.random.PRNGKey(0)

idx = (~df[['X','Y','Z','U','V','W']].isna().any(axis=1)) & (df.parallax>1)
positions = jnp.array(df.loc[idx,['X','Y','Z']].values)
velocities = jnp.array(df.loc[idx,['U','V','W']].values)
masses = jnp.ones(df.loc[idx].shape[0])

dt = 0.1
n_steps = 100

final_pos, final_vel, traj = simulate(
    positions, velocities, masses, dt, n_steps, softening=0.1
)

# --- Extract trajectory (your layout: (2, n_steps, N, 3)) ---
traj_np = np.array(traj)
positions_traj = traj_np[0]  # (n_steps, N, 3)

# --- Center-of-mass correction ---
positions_traj -= positions_traj.mean(axis=1, keepdims=True)

# --- Plot setup (ONLY ONCE) ---
fig, ax = plt.subplots()

# Initialize with first frame (fixes empty frame)
scat = ax.scatter(positions_traj[0, :, 0],
                  positions_traj[0, :, 1],
                  s=10)

lim = np.max(np.abs(positions_traj))
# ax.set_xlim(-lim, lim)
# ax.set_ylim(-lim, lim)

def update(frame):
    pos = positions_traj[frame]
    scat.set_offsets(pos[:, :2])
    ax.set_title(f"Step {frame}")
    return scat,

anim = FuncAnimation(
    fig,
    update,
    frames=positions_traj.shape[0],
    interval=30,
    blit=True
)

HTML(anim.to_jshtml())

In [ ]:
df_mem = pd.read_parquet('../data/Perren2023/UCC_members.parquet')
df_mem.tail()

In [ ]:
df_mem.shape

In [ ]:
df_mem.columns

In [ ]:
df_mem.groupby('name').count().sort_values(by='Source', ascending=False)

In [ ]:
df_mem = df_mem.rename(
    {'Plx': 'parallax',
     'RV': 'radial_velocity',
     'RA_ICRS': 'ra',
     'DE_ICRS': 'dec',
     'pmRA': 'pmra',
     'pmDE': 'pmdec',
    }, axis=1
)

In [ ]:
from wakai.catalog import get_transformed_coord

df_mem = get_transformed_coord(df_mem, frame='galactocentric')
df_mem.head()

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- Simulation ---
key = jax.random.PRNGKey(0)

cname = 'ic2602'
d = df_mem.query("name==@cname")
idx = (~d[['X','Y','Z','U','V','W']].isna().any(axis=1)) #& (df.parallax>1)
positions = jnp.array(d.loc[idx,['X','Y','Z']].values)
velocities = jnp.array(d.loc[idx,['U','V','W']].values)
masses = jnp.ones(d.loc[idx].shape[0])

dt = 0.01
n_steps = 1000

final_pos, final_vel, traj = simulate(
    positions, velocities, masses, dt, n_steps, softening=0.1
)

# --- Extract trajectory (your layout: (2, n_steps, N, 3)) ---
traj_np = np.array(traj)
positions_traj = traj_np[0]  # (n_steps, N, 3)

# --- Center-of-mass correction ---
positions_traj -= positions_traj.mean(axis=1, keepdims=True)

# --- Plot setup (ONLY ONCE) ---
fig, ax = plt.subplots()

# Initialize with first frame (fixes empty frame)
scat = ax.scatter(positions_traj[0, :, 0],
                  positions_traj[0, :, 1],
                  s=10)

lim = np.max(np.abs(positions_traj))
# ax.set_xlim(-lim, lim)
# ax.set_ylim(-lim, lim)

def update(frame):
    pos = positions_traj[frame]
    scat.set_offsets(pos[:, :2])
    ax.set_title(f"Step {frame}")
    return scat,

anim = FuncAnimation(
    fig,
    update,
    frames=positions_traj.shape[0],
    interval=30,
    blit=True
)

HTML(anim.to_jshtml())

https://ucc.ar/_clusters/ngc6791/

In [ ]:
%matplotlib inline

from wakai.catalog import plot_xyz_uvw

name = "collinder261" 
cname = name.lower().replace(' ', '')
color = 'probs'
d = df_mem.query("name==@cname")
assert len(d)>0
assert sum(d[color]>0.9)>10
idx = d[color]>0.9

fig, axs = plot_xyz_uvw(d[idx], 
                        color=color, 
                       # target_gaiaid=gaiaid, 
                       # target_label=f"TOI-{toiid}\nProb={prob*100:.1f}%", 
                       # target_color="r",
                       # target_size=500,
                       marker='.',
                       cmap="RdBu",
                       frame='galactic',
                       use_LSR=False
                     );
fig.suptitle(name)
axs = axs.flatten()
# ulims = -20,20
# vlims = -20,20
# wlims = 0, 10
# axs[3].set_xlim(*ulims); axs[3].set_ylim(*vlims)
# axs[4].set_xlim(*vlims); axs[4].set_ylim(*wlims)
# axs[5].set_xlim(*wlims); axs[5].set_ylim(*ulims)

In [ ]:
df.sort_values(by="Plx", ascending=False).iloc[10:15]

### Interactive

In [ ]:
from wakai.catalog import get_transformed_coord

d = get_transformed_coord(d)
d

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'iframe' 

fig = go.Figure(data=[
    go.Scatter3d(
        x=d['X'], y=d['Y'], z=d['Z'],
        mode='markers',
        marker=dict(size=1, color=d['Gmag'], colorscale='Viridis')
    )
])
# fig.update_layout(scene=dict(aspectmode='data'))
fig